In [ ]:
import os
import requests
import zipfile
from tqdm import tqdm

# 1. Setup Directories
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "datasets")
os.makedirs(DATA_DIR, exist_ok=True)

def download_file(url, save_path):
    if os.path.exists(save_path):
        print(f"✅ {os.path.basename(save_path)} already exists.")
        return
    
    print(f"⬇️ Downloading {url}...")
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    
    with open(save_path, 'wb') as file, tqdm(
        desc=os.path.basename(save_path),
        total=total_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for data in response.iter_content(chunk_size=1024):
            size = file.write(data)
            bar.update(size)

def unzip_file(zip_path, extract_to):
    print(f"📂 Unzipping {os.path.basename(zip_path)}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

# --- 2. DOWNLOAD COCO (Mini Version) ---
COCO_DIR = os.path.join(DATA_DIR, "COCO")
os.makedirs(COCO_DIR, exist_ok=True)

# We download the 2017 Val images (1GB) - Train is too big (19GB)
download_file("http://images.cocodataset.org/zips/val2017.zip", os.path.join(COCO_DIR, "val2017.zip"))
download_file("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", os.path.join(COCO_DIR, "annotations.zip"))

# Unzip
unzip_file(os.path.join(COCO_DIR, "val2017.zip"), COCO_DIR)
unzip_file(os.path.join(COCO_DIR, "annotations.zip"), COCO_DIR)

# --- 3. CLONE TACO ---
# Use system command for git
print("🐙 Cloning TACO...")
if not os.path.exists(os.path.join(DATA_DIR, "TACO")):
    os.system(f"git clone https://github.com/pedropro/TACO.git {os.path.join(DATA_DIR, 'TACO')}")
    
# Run TACO download script (Requires python environment)
# NOTE: You might need to install requirements inside TACO folder manually: 
# pip install -r datasets/TACO/requirements.txt
print("⚠️ IMPORTANT: Open your terminal, cd into datasets/TACO and run 'python download.py' to get the images.")

In [ ]:
import json
import os
from copy import deepcopy

# --- CONFIG ---
TARGET_CATEGORIES = [
    {"id": 1, "name": "Plastic", "supercategory": "Trash"},
    {"id": 2, "name": "Metal", "supercategory": "Trash"},
    {"id": 3, "name": "Glass", "supercategory": "Trash"},
    {"id": 4, "name": "Paper", "supercategory": "Trash"},
]

CATEGORY_MAPPING = {
    "Plastic": ["plastic", "bottle", "bag", "wrapper", "cup", "straw"],
    "Metal": ["metal", "can", "aluminum", "tin"],
    "Glass": ["glass", "jar", "broken"],
    "Paper": ["paper", "carton", "cardboard", "box", "newspaper"]
}

def get_target_id(cat_name):
    cat_name = cat_name.lower()
    for target, keywords in CATEGORY_MAPPING.items():
        for k in keywords:
            if k in cat_name:
                return next(c["id"] for c in TARGET_CATEGORIES if c["name"] == target)
    return None

# Paths (Adjust these if your folders are named differently)
DATASETS = [
    {
        "name": "TACO",
        "json": os.path.join(DATA_DIR, "TACO/data/annotations.json"),
        "img_root": os.path.abspath(os.path.join(DATA_DIR, "TACO/data"))
    },
    {
        "name": "COCO",
        "json": os.path.join(DATA_DIR, "COCO/annotations/instances_val2017.json"),
        "img_root": os.path.abspath(os.path.join(DATA_DIR, "COCO/val2017"))
    },
    {
        "name": "BePLIS",
        # Check your unzipped BePLIS folder for the correct json name!
        "json": os.path.join(DATA_DIR, "BePLIS/annotations.json"), 
        "img_root": os.path.abspath(os.path.join(DATA_DIR, "BePLIS/images"))
    }
]

merged_data = {"images": [], "annotations": [], "categories": TARGET_CATEGORIES}
ann_id_count = 1
img_id_count = 1

for ds in DATASETS:
    if not os.path.exists(ds["json"]):
        print(f"⚠️ Skipped {ds['name']}: JSON not found at {ds['json']}")
        continue
        
    print(f"Processing {ds['name']}...")
    with open(ds["json"], 'r') as f:
        data = json.load(f)

    # 1. Map Categories
    cat_map = {} # old_id -> new_id
    for cat in data['categories']:
        new_id = get_target_id(cat['name'])
        if new_id:
            cat_map[cat['id']] = new_id

    # 2. Process Images
    img_map = {} # old_img_id -> new_img_id
    for img in data['images']:
        new_img = deepcopy(img)
        new_img['id'] = img_id_count
        # Absolute path is CRITICAL for local training
        new_img['file_name'] = os.path.join(ds['img_root'], img['file_name'])
        
        merged_data['images'].append(new_img)
        img_map[img['id']] = img_id_count
        img_id_count += 1

    # 3. Process Annotations
    for ann in data['annotations']:
        if ann['category_id'] in cat_map:
            new_ann = deepcopy(ann)
            new_ann['id'] = ann_id_count
            new_ann['image_id'] = img_map[ann['image_id']]
            new_ann['category_id'] = cat_map[ann['category_id']]
            
            merged_data['annotations'].append(new_ann)
            ann_id_count += 1

# Save
OUTPUT_FILE = os.path.join(DATA_DIR, "GreenMalaysia_Master.json")
with open(OUTPUT_FILE, 'w') as f:
    json.dump(merged_data, f)

print(f"✅ Master Dataset saved to: {OUTPUT_FILE}")

In [ ]:
from ultralytics.data.converter import convert_coco

# This creates a folder 'yolo_labels/' with .txt files
convert_coco(
    labels_dir=DATA_DIR,
    save_dir=os.path.join(DATA_DIR, "yolo_labels"),
    use_segments=False,
    cls91to80=False
)

print("✅ Converted to YOLO format! You are ready to train.")